TF-IDF Text Clustering with Preprocessing

In [3]:
import re
import string
import numpy as np
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from tabulate import tabulate
from collections import Counter

dataset = [
    "I love playing football on the weekends",
    "I enjoy hiking and camping in the mountains",
    "I like to read books and watch movies",
    "I prefer playing video games over sports",
    "I love listening to music and going to concerts"
]

# Text preprocessing function
def preprocess_text(text):
    text = text.lower()                          # lowercase
    text = re.sub(r'\d+', '', text)             # remove numbers
    text = text.translate(str.maketrans('', '', string.punctuation))  # remove punctuation
    tokens = text.split()                       # tokenize
    tokens = [word for word in tokens if word not in ENGLISH_STOP_WORDS]  # remove stopwords
    return " ".join(tokens)

# Apply preprocessing
cleaned_dataset = [preprocess_text(doc) for doc in dataset]

print("Original vs Preprocessed Text:")
for original, cleaned in zip(dataset, cleaned_dataset):
    print(f"Original     : {original}")
    print(f"Preprocessed : {cleaned}\n")

# Vectorize
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(cleaned_dataset)

# Clustering
k = 2
km = KMeans(n_clusters=k, random_state=42, n_init=10)
km.fit(X)
y_pred = km.predict(X)

# Display table
table_data = [["Document", "Preprocessed Text", "Predicted Cluster"]]
table_data.extend([[doc, clean, cluster] for doc, clean, cluster in zip(dataset, cleaned_dataset, y_pred)])
print(tabulate(table_data, headers="firstrow", tablefmt="grid"))

# Print top terms per cluster
print("\nTop terms per cluster:")
order_centroids = km.cluster_centers_.argsort()[:, ::-1]
terms = vectorizer.get_feature_names_out()

for i in range(k):
    print(f"Cluster {i}:")
    for ind in order_centroids[i, :10]:
        print(terms[ind])
    print()

# Purity calculation used in the lab
total_samples = len(y_pred)
cluster_label_counts = [Counter(y_pred)]
purity = sum(max(cluster.values()) for cluster in cluster_label_counts) / total_samples
print("Purity:", purity)

Original vs Preprocessed Text:
Original     : I love playing football on the weekends
Preprocessed : love playing football weekends

Original     : I enjoy hiking and camping in the mountains
Preprocessed : enjoy hiking camping mountains

Original     : I like to read books and watch movies
Preprocessed : like read books watch movies

Original     : I prefer playing video games over sports
Preprocessed : prefer playing video games sports

Original     : I love listening to music and going to concerts
Preprocessed : love listening music going concerts

+-------------------------------------------------+-------------------------------------+---------------------+
| Document                                        | Preprocessed Text                   |   Predicted Cluster |
+=================================================+=====================================+=====================+
| I love playing football on the weekends         | love playing football weekends      |                 

Word2Vec Text Clustering with Preprocessing

In [4]:
import re
import string
import numpy as np
from sklearn.cluster import KMeans
from gensim.models import Word2Vec
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from tabulate import tabulate
from collections import Counter

dataset = [
    "I love playing football on the weekends",
    "I enjoy hiking and camping in the mountains",
    "I like to read books and watch movies",
    "I prefer playing video games over sports",
    "I love listening to music and going to concerts"
]

# Text preprocessing function
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'\d+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = text.split()
    tokens = [word for word in tokens if word not in ENGLISH_STOP_WORDS]
    return tokens

# Apply preprocessing
tokenized_dataset = [preprocess_text(doc) for doc in dataset]

print("Preprocessed Tokens:")
for doc, tokens in zip(dataset, tokenized_dataset):
    print(doc)
    print(tokens)
    print()

# Train Word2Vec model
word2vec_model = Word2Vec(
    sentences=tokenized_dataset,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4
)

# Create document embeddings by averaging word vectors
X = np.array([
    np.mean([word2vec_model.wv[word] for word in tokens if word in word2vec_model.wv], axis=0)
    for tokens in tokenized_dataset
])

# Clustering
k = 2
km = KMeans(n_clusters=k, random_state=42, n_init=10)
km.fit(X)
y_pred = km.predict(X)

# Display table
table_data = [["Document", "Tokens", "Predicted Cluster"]]
table_data.extend([[doc, " ".join(tokens), cluster] for doc, tokens, cluster in zip(dataset, tokenized_dataset, y_pred)])
print(tabulate(table_data, headers="firstrow", tablefmt="grid"))

# Purity calculation used in the lab
total_samples = len(y_pred)
cluster_label_counts = [Counter(y_pred)]
purity = sum(max(cluster.values()) for cluster in cluster_label_counts) / total_samples
print("Purity:", purity)

Preprocessed Tokens:
I love playing football on the weekends
['love', 'playing', 'football', 'weekends']

I enjoy hiking and camping in the mountains
['enjoy', 'hiking', 'camping', 'mountains']

I like to read books and watch movies
['like', 'read', 'books', 'watch', 'movies']

I prefer playing video games over sports
['prefer', 'playing', 'video', 'games', 'sports']

I love listening to music and going to concerts
['love', 'listening', 'music', 'going', 'concerts']

+-------------------------------------------------+-------------------------------------+---------------------+
| Document                                        | Tokens                              |   Predicted Cluster |
+=================================================+=====================================+=====================+
| I love playing football on the weekends         | love playing football weekends      |                   0 |
+-------------------------------------------------+-----------------------------